# 09 — Event Probability Calibration

This notebook records the development-only selection and locking of the
uniform event-probability mixing parameter.

The probabilistic temperature model and continuous dispersion scale were
selected in earlier stages and are not changed here.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data/manifests/09_probability_calibration_manifest.json").exists():
    ROOT = ROOT.parents[1]

manifest = json.loads(
    (ROOT / "data/manifests/09_probability_calibration_manifest.json")
    .read_text(encoding="utf-8")
)

grid = pd.read_csv(
    ROOT / "outputs/diagnostics/09_probability_mixing_grid_scores.csv"
)

development = pd.read_csv(
    ROOT / "outputs/diagnostics/09_development_regularised_event_probability_panel.csv"
)

locked = pd.read_csv(
    ROOT / "outputs/diagnostics/09_locked_regularised_event_probability_panel.csv"
)

checks = pd.read_csv(
    ROOT / "outputs/diagnostics/09_probability_calibration_integrity_checks.csv"
)

print("Status:", manifest["status"])
print("Selected model:", manifest["selected_model"])
print("Continuous scale:", manifest["locked_continuous_dispersion_scale"])
print("Strict winner lambda:", manifest["strict_winner_lambda"])
print("Selected one-SE lambda:", manifest["selected_uniform_mixing_lambda"])

Status: PROBABILITY_CALIBRATION_LOCKED
Selected model: pooled_empirical_residual
Continuous scale: 1.25
Strict winner lambda: 0.02
Selected one-SE lambda: 0.01


## Uniform mixing

For an eleven-event probability vector
\(\widehat{\boldsymbol p}\), define

\[
\widehat p_j^{(\lambda)}
=
(1-\lambda)\widehat p_j+\frac{\lambda}{11},
\qquad j=1,\ldots,11.
\]

The candidate grid is
\(\lambda\in\{0.00,0.01,\ldots,1.00\}\).

Scores are averaged across the four decision rules within each settlement
date. The 38 dates are treated as the uncertainty units. The selected value
is the smallest \(\lambda\) within one standard error of the best mean
development log score.

In [2]:
selected = grid.loc[
    grid["selected_lambda"].astype(str).str.lower().isin({"true", "1"})
]

eligible = grid.loc[
    grid["within_one_standard_error"].astype(str).str.lower().isin({"true", "1"})
]

assert len(grid) == 101
assert len(selected) == 1
assert float(selected["mixing_lambda"].iloc[0]) == 0.01
assert float(eligible["mixing_lambda"].min()) == 0.01

development_sums = development.groupby("row_id")[
    "regularised_event_probability"
].sum()

locked_sums = locked.groupby("row_id")[
    "regularised_event_probability"
].sum()

assert len(development_sums) == 152
assert locked["row_id"].nunique() == 159
assert locked["target_date"].nunique() == 40
assert np.allclose(development_sums, 1.0)
assert np.allclose(locked_sums, 1.0)

print("Development probability books:", len(development_sums))
print("Locked probability books:", locked["row_id"].nunique())
print("Locked dates:", locked["target_date"].nunique())

Development probability books: 152
Locked probability books: 159
Locked dates: 40


## Evidential boundary

The parameter is selected using development outcomes only.

Holdout and June outcomes do not influence selection. This notebook does not
calculate holdout or June scores, access market prices or calculate trading
returns.

In [3]:
assert manifest["holdout_outcomes_used_for_selection"] is False
assert manifest["external_test_outcomes_used_for_selection"] is False
assert manifest["holdout_scores_calculated"] is False
assert manifest["external_test_scores_calculated"] is False
assert manifest["market_prices_accessed"] is False
assert manifest["trading_returns_calculated"] is False

passed = (
    checks["passed"].astype(str).str.strip().str.lower().isin({"true", "1"})
)

assert passed.all()

print("All integrity checks passed:", True)

All integrity checks passed: True
